In [ ]:
!pip install numpy pandas matplotlib seaborn scikit-learn umap-learn astropy astroquery hdbscan




In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import hdbscan


from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import umap
from astroquery.gaia import Gaia
Gaia.ROW_LIMIT = 50000
from sklearn.neighbors import NearestNeighbors
from astropy.coordinates import SkyCoord
import astropy.units as u

plt.style.use("default")


In [ ]:
query = """
SELECT TOP 50000
    source_id,
    phot_g_mean_mag,
    bp_rp,
    parallax,
    parallax_error,
    ruwe
FROM gaiadr3.gaia_source
WHERE
    phot_g_mean_mag < 17
    AND parallax > 0
    AND bp_rp IS NOT NULL
    AND ruwe < 1.4
"""


In [ ]:
job = Gaia.launch_job(query)
results = job.get_results()
df = results.to_pandas()

print("Rows:", len(df))


In [ ]:
print("Rows:", len(df))
df.describe()


In [ ]:
df.head()


In [ ]:
df["M_G"] = df["phot_g_mean_mag"] + 5 * np.log10(df["parallax"] / 1000) + 5


In [ ]:
features = df[["bp_rp", "M_G", "parallax"]].copy()


In [ ]:
features.head()

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)


In [ ]:
pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)


In [ ]:
explained_var = pca.explained_variance_ratio_

for i, var in enumerate(explained_var, 1):
    print(f"PC{i}: {var:.3f}")


In [ ]:
plt.figure(figsize=(6,5))
plt.scatter(X_pca[:, 0], X_pca[:, 1], s=3, alpha=0.3)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA Projection of Gaia Stellar Sample")
plt.show()


In [ ]:
loadings = pd.DataFrame(
    pca.components_,
    columns=features.columns,
    index=["PC1", "PC2", "PC3"]
)
loadings


In [ ]:
reducer = umap.UMAP(
    n_neighbors=20,
    min_dist=0.1,
    n_components=2,
    random_state=42
)


In [ ]:
X_umap = reducer.fit_transform(X_scaled)


In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(
    X_umap[:, 0],
    X_umap[:, 1],
    s=3,
    alpha=0.3
)
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.title("UMAP Embedding of Gaia Stellar Sample")
plt.show()


In [ ]:
plt.figure(figsize=(6,6))
sc = plt.scatter(
    X_umap[:, 0],
    X_umap[:, 1],
    c=df["bp_rp"],
    cmap="viridis",
    s=3,
    alpha=0.5
)
plt.colorbar(sc, label="BP − RP (Color)")
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.title("UMAP Colored by Stellar Color (BP−RP)")
plt.show()


In [ ]:
plt.figure(figsize=(6,6))
sc = plt.scatter(
    X_umap[:, 0],
    X_umap[:, 1],
    c=df["M_G"],
    cmap="plasma",
    s=3,
    alpha=0.5
)
plt.colorbar(sc, label="Absolute G Magnitude (M_G)")
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.title("UMAP Colored by Absolute Magnitude")
plt.show()





## Physical Interpretation of the UMAP Embedding
### When colored by stellar color and absolute magnitude, the UMAP embedding reveals smooth and continuous gradients that align with known stellar evolutionary sequences. This demonstrates that the non-linear structure captured by the embedding reflects genuine astrophysical properties rather than algorithmic artifacts.

In [ ]:
plt.figure(figsize=(6,6))
sc = plt.scatter(
    X_umap[:, 0],
    X_umap[:, 1],
    c=df["parallax"],
    cmap="viridis",
    s=3,
    alpha=0.5
)
plt.colorbar(sc, label="Parallax (mas)")
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.title("UMAP Colored by Parallax")
plt.show()


### Survey Geometry and Selection Effects
Coloring the UMAP embedding by parallax reveals additional structure related to distance and observational selection.
While temperature and luminosity dominate the primary manifold, parallax gradients highlight how survey sensitivity and geometry influence the observed stellar distribution.
This emphasizes the importance of accounting for selection effects when interpreting data-driven structures in large surveys.


In [ ]:
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=200,
    min_samples=20
)

cluster_labels = clusterer.fit_predict(X_umap)


In [ ]:
np.unique(cluster_labels)


In [ ]:
plt.figure(figsize=(6,6))
sc = plt.scatter(
    X_umap[:, 0],
    X_umap[:, 1],
    c=cluster_labels,
    cmap="tab10",
    s=3,
    alpha=0.6
)
plt.colorbar(sc, label="HDBSCAN Cluster ID")
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.title("UMAP with Density-Based Clustering (HDBSCAN)")
plt.show()


### Density-Based Clustering
To highlight regions of enhanced density within the non-linear embedding, HDBSCAN was applied to the UMAP projection.
This approach identifies coherent overdensities while explicitly treating low-density regions as noise.
The resulting clusters trace segments of the continuous stellar manifold rather than representing discrete stellar categories,
and are therefore interpreted as exploratory indicators of population structure rather than definitive classifications.


In [ ]:
# Define rough, conservative CMD regions
main_sequence = (df["M_G"] > 4.5) & (df["bp_rp"] < 1.4)
giants = (df["M_G"] < 2.0) & (df["bp_rp"] > 0.8)

df["cmd_region"] = "Other"
df.loc[main_sequence, "cmd_region"] = "MS"
df.loc[giants, "cmd_region"] = "Giant"

df["cmd_region"].value_counts()


In [ ]:
nbrs = NearestNeighbors(n_neighbors=50).fit(X_umap)
distances, indices = nbrs.kneighbors(X_umap)


In [ ]:
consistency = []

for i in range(len(df)):
    neighbor_regions = df.iloc[indices[i]]["cmd_region"]
    own_region = df.iloc[i]["cmd_region"]

    if own_region == "Other":
        consistency.append(np.nan)
    else:
        consistency.append((neighbor_regions == own_region).mean())

df["local_consistency"] = consistency


In [ ]:
df["local_consistency"].describe()


In [ ]:
anomalies = df[
    (df["cmd_region"].isin(["MS", "Giant"])) &
    (df["local_consistency"] < 0.3)
]

len(anomalies)


In [ ]:
plt.figure(figsize=(6,6))

# Background
plt.scatter(
    X_umap[:, 0],
    X_umap[:, 1],
    s=2,
    alpha=0.2,
    color="gray"
)

# Overlay anomalies
plt.scatter(
    X_umap[anomalies.index, 0],
    X_umap[anomalies.index, 1],
    s=20,
    color="red",
    label="Inconsistent Stars"
)

plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.title("Stars with CMD–Latent Space Inconsistency")
plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(6,8))

plt.scatter(
    df["bp_rp"],
    df["M_G"],
    s=2,
    alpha=0.3,
    color="gray"
)

plt.scatter(
    anomalies["bp_rp"],
    anomalies["M_G"],
    s=20,
    color="red",
    label="Inconsistent Stars"
)

plt.gca().invert_yaxis()
plt.xlabel("BP − RP")
plt.ylabel("M_G")
plt.title("CMD Highlighting Latent-Space Inconsistent Stars")
plt.legend()
plt.show()


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10,4))

# RUWE comparison
ax[0].hist(df["ruwe"], bins=50, alpha=0.5, label="All stars")
ax[0].hist(anomalies["ruwe"], bins=50, alpha=0.7, label="Anomalies")
ax[0].set_xlabel("RUWE")
ax[0].legend()

# Parallax error comparison
ax[1].hist(df["parallax_error"], bins=50, alpha=0.5, label="All stars")
ax[1].hist(anomalies["parallax_error"], bins=50, alpha=0.7, label="Anomalies")
ax[1].set_xlabel("Parallax Error (mas)")
ax[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(6,4))
plt.hist(df["parallax"], bins=100, alpha=0.5, label="All stars")
plt.hist(anomalies["parallax"], bins=100, alpha=0.7, label="Anomalies")
plt.xlabel("Parallax (mas)")
plt.legend()
plt.title("Parallax Distribution Comparison")
plt.show()


In [ ]:
bins = np.linspace(df["bp_rp"].min(), df["bp_rp"].max(), 50)
df["color_bin"] = np.digitize(df["bp_rp"], bins)


In [ ]:
bins = np.linspace(df["bp_rp"].min(), df["bp_rp"].max(), 50)
df["color_bin"] = np.digitize(df["bp_rp"], bins)


In [ ]:
ms_ridge = (
    df[df["cmd_region"] == "MS"]
    .groupby("color_bin")["M_G"]
    .median()
)


In [ ]:
ms_ridge.head()


In [ ]:
df["ridge_offset"] = df["M_G"] - df["color_bin"].map(ms_ridge)


In [ ]:
df["ridge_offset"].describe()


In [ ]:
anomalies = df[
    (df["cmd_region"].isin(["MS", "Giant"])) &
    (df["local_consistency"] < 0.3)
].copy()


In [ ]:
anomalies[["bp_rp", "M_G", "ridge_offset"]].head()


In [ ]:
plt.figure(figsize=(6,4))
plt.hist(df["ridge_offset"], bins=100, alpha=0.5, label="All stars")
plt.hist(anomalies["ridge_offset"], bins=100, alpha=0.7, label="Anomalies")
plt.xlabel("Offset from Main-Sequence Ridge (mag)")
plt.legend()
plt.title("Deviation from Main-Sequence Ridge")
plt.show()


In [ ]:
df["abs_ridge_offset"] = np.abs(df["ridge_offset"])
anomalies["abs_ridge_offset"] = np.abs(anomalies["ridge_offset"])


In [ ]:
plt.figure(figsize=(6,4))
plt.hist(df["abs_ridge_offset"], bins=100, alpha=0.5, label="All stars", density=True)
plt.hist(anomalies["abs_ridge_offset"], bins=100, alpha=0.7, label="Anomalies", density=True)
plt.xlim(0, 2)
plt.xlabel("|Offset from MS Ridge| (mag)")
plt.legend()
plt.title("Distance from Main-Sequence Ridge")
plt.show()


In [ ]:
anomaly_ids = anomalies["source_id"].values
len(anomaly_ids)


In [ ]:
query_coords = f"""
SELECT
    source_id,
    ra,
    dec
FROM gaiadr3.gaia_source
WHERE source_id IN ({",".join(map(str, anomaly_ids[:1000]))})
"""


In [ ]:
job_coords = Gaia.launch_job(query_coords)
coords = job_coords.get_results().to_pandas()

coords.head()


In [ ]:
anomalies = anomalies.merge(coords, on="source_id", how="left")
anomalies[["ra", "dec"]].describe()


In [ ]:
plt.figure(figsize=(7,4))
plt.scatter(
    anomalies["ra"],
    anomalies["dec"],
    s=5,
    alpha=0.7
)
plt.xlabel("Right Ascension (deg)")
plt.ylabel("Declination (deg)")
plt.title("Sky Distribution of Latent-Space Anomalies")
plt.show()


In [ ]:

coords_gal = SkyCoord(
    ra=anomalies["ra"].values * u.deg,
    dec=anomalies["dec"].values * u.deg,
    frame="icrs"
).galactic

anomalies["l"] = coords_gal.l.deg
anomalies["b"] = coords_gal.b.deg


In [ ]:
plt.figure(figsize=(6,4))
plt.hist(anomalies["b"], bins=60, alpha=0.7)
plt.xlabel("Galactic Latitude b (deg)")
plt.ylabel("Count")
plt.title("Galactic Latitude Distribution of Anomalies")
plt.show()


In [ ]:
anomalies["distance_pc"] = 1000 / anomalies["parallax"]


In [ ]:
plt.figure(figsize=(6,4))
plt.hist(anomalies["distance_pc"], bins=80, alpha=0.7)
plt.xlabel("Distance (pc)")
plt.ylabel("Count")
plt.title("Distance Distribution of Anomalies")
plt.show()


In [ ]:
final_candidates = df[
    (df["cmd_region"].isin(["MS", "Giant"])) &
    (df["local_consistency"] < 0.3) &
    (df["abs_ridge_offset"] > 0.2) &
    (df["abs_ridge_offset"] < 1.0) &
    (df["ruwe"] < 1.4)
].copy()


In [ ]:
len(final_candidates)


In [ ]:
columns_to_save = [
    "source_id",
    "bp_rp",
    "M_G",
    "parallax",
    "parallax_error",
    "ruwe",
    "local_consistency",
    "abs_ridge_offset"
]

# Add coordinates if available
for col in ["ra", "dec", "l", "b", "distance_pc"]:
    if col in final_candidates.columns:
        columns_to_save.append(col)

final_candidates = final_candidates[columns_to_save]
final_candidates.head()


In [ ]:
final_candidates = final_candidates.sort_values(
    by="local_consistency",
    ascending=True
)


In [ ]:
final_candidates.to_csv(
    "anomaly_candidates_gaia_dr3.csv",
    index=False
)


In [ ]:
!ls


## Final Synthesis and Project Closure

This project set out to explore how stellar populations in Gaia DR3 organize themselves when analyzed without predefined labels, and to identify where data-driven similarity representations diverge from classical color–magnitude expectations.

Using unsupervised dimensionality reduction and local neighborhood analysis, we identified a small subset of stars whose latent-space similarity is inconsistent with their CMD-based classification. Follow-up analysis demonstrated that these sources are not dominated by poor astrometry, extreme luminosity offsets, or distance artifacts. Instead, they are statistically overrepresented in CMD transition regions, where stellar evolutionary phases and photometric properties naturally overlap.

The resulting anomaly candidate catalog represents a set of sources that highlight intrinsic degeneracies in photometry-only stellar characterization. These candidates are not claimed as new stellar populations, but rather as targets for future investigation using additional information such as spectroscopy, extinction modeling, or chemical abundances.

At this point, the analysis is intentionally concluded. Further refinement would require external data or a fundamentally different observational axis, and extending the current approach would risk over-interpretation. The goal of this project was not exhaustive classification, but to understand where unsupervised methods succeed, where they struggle, and how those limitations can be identified in a principled way.
